# US Accidents Capstone – Phase 1: Analysis and Insights

**Student:** Jackson Theuri Mururi

**Methodology:** CRISP-DM

This notebook investigates factors associated with accident severity using the sampled **500,000-record US Accidents dataset**.



## Business Understanding

Road accidents place pressure on emergency services and transport agencies. Understanding which weather conditions, times of day and road characteristics are linked to severe crashes can help stakeholders prioritize interventions.

### Stakeholders
- State Departments of Transportation
- Traffic Safety Authorities
- Emergency Response Centers

### Business Questions
1. Which weather conditions are associated with severe accidents?
2. Does accident severity change during the day?
3. Which states experience the largest number of severe crashes?
4. Which road features appear most frequently in serious accidents?


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, f_oneway

plt.rcParams["figure.figsize"]=(10,5)


In [ ]:
# Load the dataset
zip_path=r'C:\Users\PC\Documents\Data\US_Accidents_March23_sampled_500k.zip'
df=pd.read_csv(zip_path,compression='zip',low_memory=False)
print(df.shape)
df.head()


## Data Understanding

First, inspect the structure of the dataset before making any assumptions.


In [ ]:
df.info()

In [ ]:
df.describe(include='all').T.head(20)

In [ ]:
missing=df.isna().mean().sort_values(ascending=False)*100
missing.head(15)


In [ ]:
df.duplicated().sum()

## Data Preparation

Create useful features that make the analysis easier.


In [ ]:
df=df.drop_duplicates()

df['Start_Time']=pd.to_datetime(df['Start_Time'])

df['Hour']=df['Start_Time'].dt.hour
df['Month']=df['Start_Time'].dt.month
df['Day_of_Week']=df['Start_Time'].dt.day_name()

season_map={12:'Winter',1:'Winter',2:'Winter',
            3:'Spring',4:'Spring',5:'Spring',
            6:'Summer',7:'Summer',8:'Summer',
            9:'Autumn',10:'Autumn',11:'Autumn'}
df['Season']=df['Month'].map(season_map)

if 'Sunrise_Sunset' in df.columns:
    df['Is_Night']=df['Sunrise_Sunset'].eq('Night')

df['Weather_Condition']=df['Weather_Condition'].fillna('Unknown')


df.to_csv(r'C:\Users\PC\Documents\Data\us_accidents_clean.csv', index=False)
df[['Hour','Month','Season','Day_of_Week']].head()


## Exploratory Data Analysis

The goal is to understand the story hidden inside the data rather than immediately jumping into modelling.


In [ ]:
# Severity distribution
severity=df['Severity'].value_counts().sort_index()

severity.plot(kind='bar')
plt.title('Distribution of Accident Severity')
plt.xlabel('Severity')
plt.ylabel('Number of Accidents')
plt.show()


Most accidents fall into **Severity 2**, meaning the dataset is highly imbalanced. This is important because any future prediction model will need to account for this imbalance.


In [ ]:
# Accidents by hour
hourly=df.groupby('Hour').size()

hourly.plot()
plt.title('Accidents by Hour')
plt.ylabel('Accidents')
plt.show()


Rush-hour periods often stand out in traffic datasets because roads become busier, increasing opportunities for collisions.


In [ ]:
# Weather vs Severity
weather_top=df['Weather_Condition'].value_counts().head(10).index

weather_table=pd.crosstab(
    df[df['Weather_Condition'].isin(weather_top)]['Weather_Condition'],
    df[df['Weather_Condition'].isin(weather_top)]['Severity']
)

weather_table.plot(kind='bar',stacked=True)
plt.title('Top Weather Conditions by Severity')
plt.ylabel('Accidents')
plt.show()


In [ ]:
# Top states
top_states=df['State'].value_counts().head(10)

top_states.plot(kind='bar')
plt.title('Top 10 States by Number of Accidents')
plt.ylabel('Accidents')
plt.show()


In [ ]:
# Visibility by severity
if 'Visibility(mi)' in df.columns:
    df.boxplot(column='Visibility(mi)',by='Severity')
    plt.title('Visibility by Severity')
    plt.suptitle('')
    plt.show()


The visibility boxplot helps compare whether lower visibility tends to appear alongside more severe crashes.


## Statistical Analysis

Rather than relying only on charts, statistical tests provide evidence for whether observed relationships are likely to be meaningful.


In [ ]:
# Chi-Square: Weather vs Severity
weather_sample=df[df['Weather_Condition'].isin(weather_top)]

cont=pd.crosstab(weather_sample['Weather_Condition'],weather_sample['Severity'])

chi2,p,dof,expected=chi2_contingency(cont)

print("Chi-square:",round(chi2,2))
print("P-value:",p)


**Interpretation**

- If the p-value is below **0.05**, weather condition and accident severity are statistically associated.
- This does not prove weather causes severity, but it suggests a meaningful relationship worth investigating further.


In [ ]:
# ANOVA: Visibility across severity groups
if 'Visibility(mi)' in df.columns:
    groups=[
        df[df['Severity']==s]['Visibility(mi)'].dropna()
        for s in sorted(df['Severity'].unique())
    ]

    stat,p=f_oneway(*groups)

    print("F-statistic:",round(stat,2))
    print("P-value:",p)


If the ANOVA p-value is below **0.05**, average visibility differs across severity groups.


## Key Insights

Based on the exploratory analysis and statistical testing, the project should communicate findings in language that decision-makers can easily understand.

### Insight 1

Weather conditions appear to have a measurable relationship with accident severity, suggesting that weather-aware traffic management could improve preparedness.

### Insight 2

Accidents are not evenly distributed throughout the day, meaning emergency resource allocation can be adjusted around high-risk hours.

### Insight 3

Some states consistently record higher accident volumes, indicating that location-specific interventions may be more effective than one-size-fits-all strategies.


## Tableau Dashboard Plan

The dashboard should include:

- Total Accidents KPI
- Average Severity KPI
- US Accident Map
- Severity Distribution
- Weather vs Severity
- Accidents by Hour
- State Filter
- Weather Filter
- Time Filter

The dashboard should allow stakeholders to interact with the findings instead of reading static charts.


## Limitations

- The dataset is a sampled version rather than the full US Accidents dataset.
- Some weather variables contain missing values.
- The data identifies associations rather than proving causation.
- Severity is highly imbalanced, which will affect future predictive models.


## Preparing for Phase 2

This analysis naturally leads into a supervised learning problem.

**Target:** `Severity`

**Candidate Features**

- Temperature
- Visibility
- Humidity
- Wind Speed
- Weather Condition
- Hour
- Month
- State
- Traffic Signal
- Junction

Because these variables exist before or during the incident, they are suitable predictive features without obvious leakage.


## Preparing for Phase 3

The dataset contains an accident `Description` field with nearly every record populated.

Recommended Route:

**Route A – Text Classification**

Use the accident description to predict the severity category and evaluate performance using Accuracy, Precision, Recall and F1-score.
